In [41]:
import sqlite3
import pandas as pd
from pathlib import Path

In [42]:
conn = sqlite3.connect('clima_lab.db')
cursor = conn.cursor()

In [43]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS registros (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ciudad TEXT NOT NULL,
    temp_c REAL,
    humedad INTEGER,
    precip_mm REAL,
    viento_kmh REAL,
    fecha TEXT
)
''')

In [44]:
conn.commit()
cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='registros'")
print("Tabla creada:")
print(cursor.fetchone()[0])
print("\nColumnas:")
cursor.execute("PRAGMA table_info(registros)")
for col in cursor.fetchall():
    print(f"{col[1]} {col[2]}")

conn.close()

Tabla creada:
CREATE TABLE registros (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                ciudad TEXT NOT NULL,
                temp_c REAL,
                humedad INTEGER,
                precip_mm REAL,
                viento_kmh REAL,
                fecha TEXT
            )

Columnas:
id INTEGER
ciudad TEXT
temp_c REAL
humedad INTEGER
precip_mm REAL
viento_kmh REAL
fecha TEXT


In [45]:
conn = sqlite3.connect('clima_lab.db')
cursor = conn.cursor()
registro = ('Santiago', 18.5, 72, 1.2, 15.3, '2026-06-15')

In [46]:
cursor.execute('''
INSERT INTO registros (ciudad,temp_c,humedad,precip_mm,viento_kmh,fecha)
VALUES (?,?,?,?,?,?)
''', registro)
print("ID insertado:", cursor.lastrowid)

ID insertado: 11


In [47]:
datos = [
    ('Valparaíso', 15.2, 80, 3.5, 22.1, '2026-06-15'),
    ('Concepción', 12.8, 85, 8.0, 18.7, '2026-06-15'),
    ('Temuco', 9.3, 90, 12.5, 25.0, '2026-06-15'),
    ('La Serena', 17.8, 55, 0.0, 12.4, '2026-06-15')
]

In [48]:
cursor.executemany('''
INSERT INTO registros (ciudad,temp_c,humedad,precip_mm,viento_kmh,fecha)
VALUES (?,?,?,?,?,?)
''', datos)
conn.commit()

In [49]:
cursor.execute("SELECT COUNT(*) FROM registros")
print("Total registros:", cursor.fetchone()[0])

conn.close()

Total registros: 13


In [50]:
conn = sqlite3.connect('clima_lab.db')
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

print("CONSULTA 1")
cursor.execute("SELECT ciudad,temp_c,fecha FROM registros")
for fila in cursor.fetchall():
    print(f"{fila['ciudad']:12} {fila['temp_c']:5} {fila['fecha']}")

print("\nCONSULTA 2")
cursor.execute("SELECT ciudad,temp_c FROM registros ORDER BY temp_c DESC LIMIT 1")
fila = cursor.fetchone()
print(f"Ciudad más cálida: {fila['ciudad']} ({fila['temp_c']}°C)")

print("\nCONSULTA 3")
cursor.execute("SELECT ciudad,temp_c FROM registros WHERE temp_c < ?", (13,))
for fila in cursor.fetchall():
    print(fila['ciudad'], fila['temp_c'])

print("\nCONSULTA 4")
cursor.execute('''
SELECT ciudad,
AVG(temp_c) promedio_temp,
MIN(humedad) humedad_min,
MAX(humedad) humedad_max
FROM registros
GROUP BY ciudad
ORDER BY promedio_temp DESC
''')

for fila in cursor.fetchall():
    print(dict(fila))

conn.close()


CONSULTA 1
Santiago      18.5 2026-06-15
Valparaíso    16.0 2026-06-15
Concepción    12.8 2026-06-15
La Serena     17.8 2026-06-15
Santiago      18.5 2026-06-15
Valparaíso    16.0 2026-06-15
Concepción    12.8 2026-06-15
La Serena     17.8 2026-06-15
Santiago      18.5 2026-06-15
Valparaíso    15.2 2026-06-15
Concepción    12.8 2026-06-15
Temuco         9.3 2026-06-15
La Serena     17.8 2026-06-15

CONSULTA 2
Ciudad más cálida: Santiago (18.5°C)

CONSULTA 3
Concepción 12.8
Concepción 12.8
Concepción 12.8
Temuco 9.3

CONSULTA 4
{'ciudad': 'Santiago', 'promedio_temp': 18.5, 'humedad_min': 72, 'humedad_max': 72}
{'ciudad': 'La Serena', 'promedio_temp': 17.8, 'humedad_min': 55, 'humedad_max': 55}
{'ciudad': 'Valparaíso', 'promedio_temp': 15.733333333333334, 'humedad_min': 80, 'humedad_max': 80}
{'ciudad': 'Concepción', 'promedio_temp': 12.800000000000002, 'humedad_min': 85, 'humedad_max': 85}
{'ciudad': 'Temuco', 'promedio_temp': 9.3, 'humedad_min': 90, 'humedad_max': 90}


In [51]:
try:
    conn = sqlite3.connect('clima_lab.db')
    cursor = conn.cursor()

    cursor.execute(
        "UPDATE registros SET temp_c=? WHERE ciudad=?",
        (16.0, 'Valparaíso')
    )
    print("UPDATE 1:", cursor.rowcount, "fila(s)")

    cursor.execute(
        "SELECT temp_c FROM registros WHERE ciudad=?",
        ('Valparaíso',)
    )
    print("Nueva temperatura:", cursor.fetchone()[0])

    cursor.execute(
        "UPDATE registros SET humedad = humedad + 5 WHERE temp_c < ?",
        (10,)
    )
    print("UPDATE 2:", cursor.rowcount, "fila(s)")

    cursor.execute(
        "DELETE FROM registros WHERE ciudad=?",
        ('Temuco',)
    )
    print("DELETE:", cursor.rowcount, "fila(s)")

    cursor.execute(
        "SELECT COUNT(*) FROM registros WHERE ciudad=?",
        ('Temuco',)
    )
    print("Registros Temuco:", cursor.fetchone()[0])

    conn.commit()

except Exception as e:
    print("Error:", e)

finally:
    conn.close()

UPDATE 1: 3 fila(s)
Nueva temperatura: 16.0
UPDATE 2: 1 fila(s)
DELETE: 1 fila(s)
Registros Temuco: 0


In [52]:
ruta_csv = 'S15_registros_climaticos.csv'
df = pd.read_csv(ruta_csv)

In [53]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ciudad      208 non-null    object 
 1   temp_c      194 non-null    float64
 2   humedad     198 non-null    float64
 3   precip_mm   208 non-null    float64
 4   viento_kmh  208 non-null    float64
 5   fecha       208 non-null    object 
dtypes: float64(4), object(2)
memory usage: 9.9+ KB


ciudad         0
temp_c        14
humedad       10
precip_mm      0
viento_kmh     0
fecha          0
dtype: int64

In [54]:
filas_originales = len(df)
df['ciudad'] = df['ciudad'].str.title()
df['fecha'] = pd.to_datetime(
    df['fecha'],
    errors='coerce',
    dayfirst=True
).dt.strftime('%Y-%m-%d')
df = df.drop_duplicates(subset=['ciudad', 'fecha'])
df = df[(df['temp_c'] >= -5) & (df['temp_c'] <= 50)]

/tmp/nix-shell-28248-2808685565/ipykernel_31436/1548322823.py:3: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['fecha'] = pd.to_datetime(


In [55]:
filas_limpias = len(df)
eliminadas = filas_originales - filas_limpias
conn = sqlite3.connect('clima_lab.db')

In [56]:
df.to_sql(
    'registros_csv',
    conn,
    if_exists='replace',
    index=False
)

176

In [57]:
q1 = pd.read_sql_query('''
SELECT ciudad, AVG(temp_c) AS promedio
FROM registros_csv
GROUP BY ciudad
ORDER BY promedio DESC
LIMIT 1
''', conn)

In [58]:
q2 = pd.read_sql_query('''
SELECT ciudad, AVG(precip_mm) AS promedio_precipitacion
FROM registros_csv
WHERE substr(fecha,1,7) BETWEEN '2026-01' AND '2026-03'
GROUP BY ciudad
''', conn)

In [59]:
q3 = pd.read_sql_query('''
SELECT fecha, ciudad, temp_c
FROM registros_csv
ORDER BY temp_c DESC
LIMIT 5
''', conn)

In [60]:
print("Filas originales:", filas_originales)
print("Filas limpias:", filas_limpias)
print("Filas eliminadas:", eliminadas)
print("Ciudad con mayor temperatura promedio")
print(q1)
print("Promedio precipitación enero-marzo 2026")
print(q2)
print("Top 5 temperaturas")
print(q3)
conn.close()

Filas originales: 208
Filas limpias: 176
Filas eliminadas: 32
Ciudad con mayor temperatura promedio
        ciudad   promedio
0  Antofagasta  20.673333
Promedio precipitación enero-marzo 2026
         ciudad  promedio_precipitacion
0   Antofagasta                3.314286
1    Concepción                3.075000
2       Iquique                2.470000
3     La Serena                3.514286
4  Puerto Montt                3.200000
5      Rancagua                3.620000
6      Santiago                7.125000
7         Talca                3.400000
8        Temuco                3.433333
9    Valparaíso                2.130000
Top 5 temperaturas
        fecha       ciudad  temp_c
0  2026-04-25  Antofagasta    31.7
1  2026-03-09      Iquique    27.6
2  2026-01-15  Antofagasta    26.5
3  2026-05-23    La Serena    24.5
4  2026-02-18  Antofagasta    24.2
